In [1]:
# Load data to the dataframe as a starting point to create the gold layer
df = spark.read.table("Sales.sales_silver")

StatementMeta(, cf654bfc-b71f-4bb5-bb4b-8bae0ff45e6b, 3, Finished, Available, Finished)

**Create date dimension table**

In [2]:
from pyspark.sql.types import *
from delta.tables import*
    
# Define the schema for the dimdate_gold table
DeltaTable.createIfNotExists(spark) \
    .tableName("sales.dimdate_gold") \
    .addColumn("OrderDate", DateType()) \
    .addColumn("Day", IntegerType()) \
    .addColumn("Month", IntegerType()) \
    .addColumn("Year", IntegerType()) \
    .addColumn("mmmyyyy", StringType()) \
    .addColumn("yyyymm", StringType()) \
    .execute()

StatementMeta(, cf654bfc-b71f-4bb5-bb4b-8bae0ff45e6b, 4, Finished, Available, Finished)

**Create a dataframe date dimension, dimdate_gold**

In [3]:
from pyspark.sql.functions import col, dayofmonth, month, year, date_format
    
# Create dataframe for dimDate_gold
    
dfdimDate_gold = df.dropDuplicates(["OrderDate"]).select(col("OrderDate"), \
        dayofmonth("OrderDate").alias("Day"), \
        month("OrderDate").alias("Month"), \
        year("OrderDate").alias("Year"), \
        date_format(col("OrderDate"), "MMM-yyyy").alias("mmmyyyy"), \
        date_format(col("OrderDate"), "yyyyMM").alias("yyyymm"), \
    ).orderBy("OrderDate")

# Display the first 10 rows of the dataframe to preview your data

display(dfdimDate_gold.head(10))

StatementMeta(, cf654bfc-b71f-4bb5-bb4b-8bae0ff45e6b, 5, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, e49f93bf-bff4-4042-8742-b755c537d0d6)

**Update the date dimension as new data comes in**

In [4]:
from delta.tables import *
    
deltaTable = DeltaTable.forPath(spark, 'Tables/dimdate_gold')
    
dfUpdates = dfdimDate_gold
    
deltaTable.alias('gold') \
  .merge(
    dfUpdates.alias('updates'),
    'gold.OrderDate = updates.OrderDate'
  ) \
   .whenMatchedUpdate(set =
    {
          
    }
  ) \
 .whenNotMatchedInsert(values =
    {
      "OrderDate": "updates.OrderDate",
      "Day": "updates.Day",
      "Month": "updates.Month",
      "Year": "updates.Year",
      "mmmyyyy": "updates.mmmyyyy",
      "yyyymm": "updates.yyyymm"
    }
  ) \
  .execute()

StatementMeta(, cf654bfc-b71f-4bb5-bb4b-8bae0ff45e6b, 6, Finished, Available, Finished)

In [5]:
%%sql

SELECT * FROM Sales.dimdate_gold LIMIT 1000

StatementMeta(, cf654bfc-b71f-4bb5-bb4b-8bae0ff45e6b, 7, Finished, Available, Finished)

<Spark SQL result set with 914 rows and 6 fields>

**The customer dimension table**

In [6]:
from pyspark.sql.types import *
from delta.tables import *
    
# Create customer_gold dimension delta table
DeltaTable.createIfNotExists(spark) \
    .tableName("sales.dimcustomer_gold") \
    .addColumn("CustomerName", StringType()) \
    .addColumn("Email",  StringType()) \
    .addColumn("First", StringType()) \
    .addColumn("Last", StringType()) \
    .addColumn("CustomerID", LongType()) \
    .execute()

StatementMeta(, cf654bfc-b71f-4bb5-bb4b-8bae0ff45e6b, 8, Finished, Available, Finished)

**Drop duplicate customers, select specific columns, and split the “CustomerName” column to create “First” and “Last” name columns**

In [7]:
from pyspark.sql.functions import *

# Create customer_silver dataframe
dfdimCustomer_silver = (
    df.dropDuplicates(["CustomerName", "Email"])
      .select(
          col("CustomerName"),
          col("Email")
      )
      .withColumn("First", split(col("CustomerName"), " ").getItem(0))
      .withColumn("Last", split(col("CustomerName"), " ").getItem(1))
)

# Display the first 10 rows of the dataframe
display(dfdimCustomer_silver.limit(10))


StatementMeta(, cf654bfc-b71f-4bb5-bb4b-8bae0ff45e6b, 9, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, e2005086-dd34-4ee4-8c6d-4a7e704b7a20)

**Create the ID column for our customers**

In [8]:
from pyspark.sql.functions import monotonically_increasing_id, col, when, coalesce, max, lit
    
dfdimCustomer_temp = spark.read.table("Sales.dimCustomer_gold")
    
MAXCustomerID = dfdimCustomer_temp.select(coalesce(max(col("CustomerID")),lit(0)).alias("MAXCustomerID")).first()[0]
    
dfdimCustomer_gold = dfdimCustomer_silver.join(dfdimCustomer_temp,(dfdimCustomer_silver.CustomerName == dfdimCustomer_temp.CustomerName) & (dfdimCustomer_silver.Email == dfdimCustomer_temp.Email), "left_anti")
    
dfdimCustomer_gold = dfdimCustomer_gold.withColumn("CustomerID",monotonically_increasing_id() + MAXCustomerID + 1)

# Display the first 10 rows of the dataframe to preview your data

display(dfdimCustomer_gold.head(10))

StatementMeta(, cf654bfc-b71f-4bb5-bb4b-8bae0ff45e6b, 10, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 6d106810-4519-4211-ae16-30874da61d10)

**Customer table remains up-to-date as new data comes in**

In [9]:
from delta.tables import *

deltaTable = DeltaTable.forPath(spark, 'Tables/dimcustomer_gold')
    
dfUpdates = dfdimCustomer_gold
    
deltaTable.alias('gold') \
  .merge(
    dfUpdates.alias('updates'),
    'gold.CustomerName = updates.CustomerName AND gold.Email = updates.Email'
  ) \
   .whenMatchedUpdate(set =
    {
          
    }
  ) \
 .whenNotMatchedInsert(values =
    {
      "CustomerName": "updates.CustomerName",
      "Email": "updates.Email",
      "First": "updates.First",
      "Last": "updates.Last",
      "CustomerID": "updates.CustomerID"
    }
  ) \
  .execute()

StatementMeta(, cf654bfc-b71f-4bb5-bb4b-8bae0ff45e6b, 11, Finished, Available, Finished)

In [10]:
%%sql

SELECT * FROM Sales.dimcustomer_gold LIMIT 1000

StatementMeta(, cf654bfc-b71f-4bb5-bb4b-8bae0ff45e6b, 12, Finished, Available, Finished)

<Spark SQL result set with 1000 rows and 5 fields>

**Create product dimension**

In [11]:
from pyspark.sql.types import *
from delta.tables import *
    
DeltaTable.createIfNotExists(spark) \
    .tableName("sales.dimproduct_gold") \
    .addColumn("ItemName", StringType()) \
    .addColumn("ItemID", LongType()) \
    .addColumn("ItemInfo", StringType()) \
    .execute()

StatementMeta(, cf654bfc-b71f-4bb5-bb4b-8bae0ff45e6b, 13, Finished, Available, Finished)

**Transform Data call as dimproduct_silver**

In [12]:
from pyspark.sql.functions import col, split, lit, when
    
# Create product_silver dataframe
    
dfdimProduct_silver = df.dropDuplicates(["Item"]).select(col("Item")) \
    .withColumn("ItemName",split(col("Item"), ", ").getItem(0)) \
    .withColumn("ItemInfo",when((split(col("Item"), ", ").getItem(1).isNull() | (split(col("Item"), ", ").getItem(1)=="")),lit("")).otherwise(split(col("Item"), ", ").getItem(1))) 
    
# Display the first 10 rows of the dataframe to preview your data

display(dfdimProduct_silver.head(10))

StatementMeta(, cf654bfc-b71f-4bb5-bb4b-8bae0ff45e6b, 14, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, bdd3b67d-856b-468a-9bd2-f18aee29191d)

**create IDs for your dimProduct_gold table from dimProduct_silver**

In [13]:
from pyspark.sql.functions import monotonically_increasing_id, col, lit, max, coalesce
    
#dfdimProduct_temp = dfdimProduct_silver
dfdimProduct_temp = spark.read.table("Sales.dimProduct_gold")
    
MAXProductID = dfdimProduct_temp.select(coalesce(max(col("ItemID")),lit(0)).alias("MAXItemID")).first()[0]
    
dfdimProduct_gold = dfdimProduct_silver.join(dfdimProduct_temp,(dfdimProduct_silver.ItemName == dfdimProduct_temp.ItemName) & (dfdimProduct_silver.ItemInfo == dfdimProduct_temp.ItemInfo), "left_anti")
    
dfdimProduct_gold = dfdimProduct_gold.withColumn("ItemID",monotonically_increasing_id() + MAXProductID + 1)
    
# Display the first 10 rows of the dataframe to preview your data

display(dfdimProduct_gold.head(10))

StatementMeta(, cf654bfc-b71f-4bb5-bb4b-8bae0ff45e6b, 15, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 7315ab02-4c21-4e25-8a67-dd812668edcd)

**Product table remains up-to-date as new data comes in**

In [14]:
from delta.tables import *
    
deltaTable = DeltaTable.forPath(spark, 'Tables/dimproduct_gold')
            
dfUpdates = dfdimProduct_gold
            
deltaTable.alias('gold') \
  .merge(
        dfUpdates.alias('updates'),
        'gold.ItemName = updates.ItemName AND gold.ItemInfo = updates.ItemInfo'
        ) \
        .whenMatchedUpdate(set =
        {
               
        }
        ) \
        .whenNotMatchedInsert(values =
         {
          "ItemName": "updates.ItemName",
          "ItemInfo": "updates.ItemInfo",
          "ItemID": "updates.ItemID"
          }
          ) \
          .execute()

StatementMeta(, cf654bfc-b71f-4bb5-bb4b-8bae0ff45e6b, 16, Finished, Available, Finished)

In [15]:
%%sql

SELECT * FROM Sales.dimproduct_gold LIMIT 1000

StatementMeta(, cf654bfc-b71f-4bb5-bb4b-8bae0ff45e6b, 17, Finished, Available, Finished)

<Spark SQL result set with 130 rows and 3 fields>

**Create the fact table**

In [16]:
from pyspark.sql.types import *
from delta.tables import *
    
DeltaTable.createIfNotExists(spark) \
    .tableName("sales.factsales_gold") \
    .addColumn("CustomerID", LongType()) \
    .addColumn("ItemID", LongType()) \
    .addColumn("OrderDate", DateType()) \
    .addColumn("Quantity", IntegerType()) \
    .addColumn("UnitPrice", FloatType()) \
    .addColumn("Tax", FloatType()) \
    .execute()

StatementMeta(, cf654bfc-b71f-4bb5-bb4b-8bae0ff45e6b, 18, Finished, Available, Finished)

**Transform data**

In [17]:
from pyspark.sql.functions import col
    
dfdimCustomer_temp = spark.read.table("Sales.dimCustomer_gold")
dfdimProduct_temp = spark.read.table("Sales.dimProduct_gold")
    
df = df.withColumn("ItemName",split(col("Item"), ", ").getItem(0)) \
    .withColumn("ItemInfo",when((split(col("Item"), ", ").getItem(1).isNull() | (split(col("Item"), ", ").getItem(1)=="")),lit("")).otherwise(split(col("Item"), ", ").getItem(1))) \
    
    
# Create Sales_gold dataframe
    
dffactSales_gold = df.alias("df1").join(dfdimCustomer_temp.alias("df2"),(df.CustomerName == dfdimCustomer_temp.CustomerName) & (df.Email == dfdimCustomer_temp.Email), "left") \
        .join(dfdimProduct_temp.alias("df3"),(df.ItemName == dfdimProduct_temp.ItemName) & (df.ItemInfo == dfdimProduct_temp.ItemInfo), "left") \
    .select(col("df2.CustomerID") \
        , col("df3.ItemID") \
        , col("df1.OrderDate") \
        , col("df1.Quantity") \
        , col("df1.UnitPrice") \
        , col("df1.Tax") \
    ).orderBy(col("df1.OrderDate"), col("df2.CustomerID"), col("df3.ItemID"))
    
# Display the first 10 rows of the dataframe to preview your data
    
display(dffactSales_gold.head(10))

StatementMeta(, cf654bfc-b71f-4bb5-bb4b-8bae0ff45e6b, 19, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, b8a695ca-e07a-43a3-8b58-51e0cedf1ab5)

Here you’re using Delta Lake’s merge operation to synchronize and update the factsales_gold table with new sales data (dffactSales_gold). The operation compares the order date, customer ID, and item ID between the existing data (silver table) and the new data (updates DataFrame), updating matching records and inserting new records as needed.

In [18]:
from delta.tables import *
    
deltaTable = DeltaTable.forPath(spark, 'Tables/factsales_gold')
    
dfUpdates = dffactSales_gold
    
deltaTable.alias('gold') \
  .merge(
    dfUpdates.alias('updates'),
    'gold.OrderDate = updates.OrderDate AND gold.CustomerID = updates.CustomerID AND gold.ItemID = updates.ItemID'
  ) \
   .whenMatchedUpdate(set =
    {
          
    }
  ) \
 .whenNotMatchedInsert(values =
    {
      "CustomerID": "updates.CustomerID",
      "ItemID": "updates.ItemID",
      "OrderDate": "updates.OrderDate",
      "Quantity": "updates.Quantity",
      "UnitPrice": "updates.UnitPrice",
      "Tax": "updates.Tax"
    }
  ) \
  .execute()

StatementMeta(, cf654bfc-b71f-4bb5-bb4b-8bae0ff45e6b, 20, Finished, Available, Finished)

In [19]:
%%sql

SELECT * FROM Sales.factsales_gold LIMIT 1000

StatementMeta(, cf654bfc-b71f-4bb5-bb4b-8bae0ff45e6b, 21, Finished, Available, Finished)

<Spark SQL result set with 1000 rows and 6 fields>